# Claussifier - Stage 1: BERT Training Pipeline

**Goal:** Build complete preprocessing pipeline, train BERT model, and evaluate performance

**What we'll accomplish:**
- Data preprocessing with BERT tokenizer
- Multi-label encoding with class weights
- BERT model setup for multi-label classification
- Training loop with progress tracking
- Comprehensive evaluation metrics
- Model checkpointing to Google Drive

---

## 1. Setup & Installation

In [ ]:
# Install required libraries
!pip install -q datasets transformers torch scikit-learn accelerate

In [ ]:
# Import libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW # Corrected import for AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, multilabel_confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
import json
from datetime import datetime

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("✓ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set up directories
PROJECT_DIR = "/content/drive/MyDrive/Claussifier"
MODEL_DIR = f"{PROJECT_DIR}/models"
RESULTS_DIR = f"{PROJECT_DIR}/results"

!mkdir -p "{MODEL_DIR}"
!mkdir -p "{RESULTS_DIR}"

print("✓ Google Drive mounted and directories ready!")

## 3. Configuration & Hyperparameters

In [ ]:
# Configuration
CONFIG = {
    # Model
    'model_name': 'bert-base-uncased',
    'num_labels': 8,
    'max_length': 512,
    
    # Training
    'batch_size': 16,
    'num_epochs': 5,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'warmup_steps': 500,
    'max_grad_norm': 1.0,
    
    # Evaluation
    'eval_every_n_steps': 100,
    'save_best_model': True,
    
    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Random seed
    'seed': 42
}

# Label names
LABEL_NAMES = [
    "Limitation of liability",
    "Unilateral termination",
    "Unilateral change",
    "Content removal",
    "Contract by using",
    "Choice of law",
    "Jurisdiction",
    "Arbitration"
]

# Class weights (from Day 1 EDA)
CLASS_WEIGHTS = torch.tensor([
    3.62, 4.97, 5.67, 9.47, 9.10, 17.73, 20.34, 24.70
])

# Set random seed for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

print(f"\nClass weights: {CLASS_WEIGHTS.tolist()}")

## 4. Load Dataset

In [ ]:
# Load dataset from Hugging Face
print("Loading LexGLUE unfair_tos dataset...")
dataset = load_dataset("lex_glue", "unfair_tos")

print("\n✓ Dataset loaded successfully!")
print(f"Train: {len(dataset['train'])} examples")
print(f"Validation: {len(dataset['validation'])} examples")
print(f"Test: {len(dataset['test'])} examples")

## 5. Data Preprocessing

Create custom Dataset class for BERT tokenization and multi-label encoding.

In [ ]:
class ClauseDataset(Dataset):
    """Custom Dataset for legal clause classification."""
    
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label_list = self.labels[idx]
        
        # Tokenize text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        # Convert labels to multi-hot encoding
        label_tensor = torch.zeros(8, dtype=torch.float)
        for label_idx in label_list:
            label_tensor[label_idx] = 1.0
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': label_tensor
        }

print("✓ ClauseDataset class defined!")

In [ ]:
# Initialize tokenizer
print(f"Loading tokenizer: {CONFIG['model_name']}")
tokenizer = BertTokenizer.from_pretrained(CONFIG['model_name'])

# Extract texts and labels
train_texts = dataset['train']['text']
train_labels = dataset['train']['labels']

val_texts = dataset['validation']['text']
val_labels = dataset['validation']['labels']

test_texts = dataset['test']['text']
test_labels = dataset['test']['labels']

# Create datasets
train_dataset = ClauseDataset(train_texts, train_labels, tokenizer, CONFIG['max_length'])
val_dataset = ClauseDataset(val_texts, val_labels, tokenizer, CONFIG['max_length'])
test_dataset = ClauseDataset(test_texts, test_labels, tokenizer, CONFIG['max_length'])

print("\n✓ Datasets created!")
print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")
print(f"Test dataset: {len(test_dataset)} examples")

In [ ]:
# Test the dataset
print("Testing dataset...\n")
sample = train_dataset[0]
print(f"Input IDs shape: {sample['input_ids'].shape}")
print(f"Attention mask shape: {sample['attention_mask'].shape}")
print(f"Labels shape: {sample['labels'].shape}")
print(f"Labels: {sample['labels']}")
print(f"\nDecoded text (first 100 chars): {tokenizer.decode(sample['input_ids'][:100])}")

## 6. Create DataLoaders

In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✓ DataLoaders created!")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 7. Initialize BERT Model

In [ ]:
# Load BERT model for multi-label classification
print(f"Loading BERT model: {CONFIG['model_name']}")

model = BertForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=CONFIG['num_labels'],
    problem_type="multi_label_classification"
)

# Move model to device
device = torch.device(CONFIG['device'])
model.to(device)

# Move class weights to device
class_weights = CLASS_WEIGHTS.to(device)

print(f"\n✓ Model loaded and moved to {device}!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 8. Setup Training Components

In [ ]:
# Loss function with class weights
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# Learning rate scheduler
total_steps = len(train_loader) * CONFIG['num_epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CONFIG['warmup_steps'],
    num_training_steps=total_steps
)

print("✓ Training components initialized!")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {CONFIG['warmup_steps']}")

## 9. Evaluation Functions

In [ ]:
def evaluate_model(model, dataloader, device, threshold=0.5):
    """Evaluate model on a dataset."""
    model.eval()
    
    all_preds = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Compute loss
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            # Get predictions
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    # Concatenate all batches
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    # Compute metrics
    avg_loss = total_loss / len(dataloader)
    
    # Macro metrics (average across classes)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    macro_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    macro_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    
    # Micro metrics (aggregate across all instances)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    micro_precision = precision_score(all_labels, all_preds, average='micro', zero_division=0)
    micro_recall = recall_score(all_labels, all_preds, average='micro', zero_division=0)
    
    # Per-class metrics
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    per_class_precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    per_class_recall = recall_score(all_labels, all_preds, average=None, zero_division=0)
    
    return {
        'loss': avg_loss,
        'macro_f1': macro_f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'micro_f1': micro_f1,
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'per_class_f1': per_class_f1,
        'per_class_precision': per_class_precision,
        'per_class_recall': per_class_recall,
        'predictions': all_preds,
        'labels': all_labels
    }

print("✓ Evaluation function defined!")

## 10. Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_macro_f1': [],
    'val_micro_f1': [],
    'learning_rate': []
}

best_val_f1 = 0.0
start_time = datetime.now()

print("="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device: {device}")
print(f"Epochs: {CONFIG['num_epochs']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"Learning rate: {CONFIG['learning_rate']}")
print("="*80)

for epoch in range(CONFIG['num_epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['num_epochs']}")
    print("-" * 80)
    
    # Training phase
    model.train()
    train_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Training Epoch {epoch + 1}")
    for batch_idx, batch in enumerate(pbar):
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Compute loss
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
        
        # Update weights
        optimizer.step()
        scheduler.step()
        
        # Track loss
        train_loss += loss.item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}"
        })
    
    # Average training loss
    avg_train_loss = train_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)
    history['learning_rate'].append(scheduler.get_last_lr()[0])
    
    # Validation phase
    print("\nValidating...")
    val_metrics = evaluate_model(model, val_loader, device)
    
    history['val_loss'].append(val_metrics['loss'])
    history['val_macro_f1'].append(val_metrics['macro_f1'])
    history['val_micro_f1'].append(val_metrics['micro_f1'])
    
    # Print epoch summary
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {val_metrics['loss']:.4f}")
    print(f"  Val Macro F1: {val_metrics['macro_f1']:.4f}")
    print(f"  Val Macro Precision: {val_metrics['macro_precision']:.4f}")
    print(f"  Val Macro Recall: {val_metrics['macro_recall']:.4f}")
    print(f"  Val Micro F1: {val_metrics['micro_f1']:.4f}")
    
    # Save best model
    if val_metrics['macro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['macro_f1']
        
        # Save model checkpoint
        checkpoint_path = f"{MODEL_DIR}/best_model_epoch{epoch+1}_f1{best_val_f1:.4f}.pt"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_f1': best_val_f1,
            'config': CONFIG
        }, checkpoint_path)
        
        print(f"\n✓ New best model saved! (F1: {best_val_f1:.4f})")
        print(f"  Saved to: {checkpoint_path}")

end_time = datetime.now()
training_duration = end_time - start_time

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total duration: {training_duration}")
print(f"Best validation F1: {best_val_f1:.4f}")
print("="*80)

## 11. Visualize Training Progress

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, CONFIG['num_epochs'] + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1 Scores
axes[1].plot(epochs_range, history['val_macro_f1'], 'g-', label='Macro F1', linewidth=2, marker='o')
axes[1].plot(epochs_range, history['val_micro_f1'], 'orange', label='Micro F1', linewidth=2, marker='s')
axes[1].axhline(y=0.65, color='red', linestyle='--', label='Target (0.65)', linewidth=1)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('Validation F1 Scores', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].plot(epochs_range, history['learning_rate'], 'purple', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[2].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training curves saved!")

## 12. Load Best Model and Evaluate on Test Set

In [ ]:
# Find best model checkpoint
import glob
checkpoints = glob.glob(f"{MODEL_DIR}/best_model_*.pt")
if checkpoints:
    # Get the latest checkpoint (highest F1)
    best_checkpoint = max(checkpoints, key=lambda x: float(x.split('_f1')[1].split('.pt')[0]))
    print(f"Loading best model from: {best_checkpoint}")
    
    # Load checkpoint
    checkpoint = torch.load(best_checkpoint)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    print(f"✓ Best model loaded (Epoch {checkpoint['epoch']}, Val F1: {checkpoint['val_f1']:.4f})")
else:
    print("⚠️ No checkpoint found, using current model")

In [ ]:
# Evaluate on test set
print("Evaluating on test set...\n")
test_metrics = evaluate_model(model, test_loader, device)

print("="*80)
print("TEST SET RESULTS")
print("="*80)
print(f"Test Loss: {test_metrics['loss']:.4f}")
print(f"\nMacro Metrics:")
print(f"  F1:        {test_metrics['macro_f1']:.4f}")
print(f"  Precision: {test_metrics['macro_precision']:.4f}")
print(f"  Recall:    {test_metrics['macro_recall']:.4f}")
print(f"\nMicro Metrics:")
print(f"  F1:        {test_metrics['micro_f1']:.4f}")
print(f"  Precision: {test_metrics['micro_precision']:.4f}")
print(f"  Recall:    {test_metrics['micro_recall']:.4f}")
print("="*80)

## 13. Per-Class Performance Analysis

In [ ]:
# Create per-class performance table
per_class_df = pd.DataFrame({
    'Risk Category': LABEL_NAMES,
    'F1': test_metrics['per_class_f1'],
    'Precision': test_metrics['per_class_precision'],
    'Recall': test_metrics['per_class_recall']
})

# Sort by F1 score
per_class_df = per_class_df.sort_values('F1', ascending=False).reset_index(drop=True)

print("\nPer-Class Performance on Test Set:\n")
print(per_class_df.to_string(index=False))

# Save to CSV
per_class_df.to_csv(f'{RESULTS_DIR}/per_class_performance.csv', index=False)
print("\n✓ Per-class performance saved to CSV")

In [ ]:
# Visualize per-class performance
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(LABEL_NAMES))
width = 0.25

ax.bar(x - width, test_metrics['per_class_precision'], width, label='Precision', alpha=0.8)
ax.bar(x, test_metrics['per_class_recall'], width, label='Recall', alpha=0.8)
ax.bar(x + width, test_metrics['per_class_f1'], width, label='F1', alpha=0.8)

ax.set_xlabel('Risk Category', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Performance on Test Set', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Per-class performance chart saved!")

## 14. Save Final Model and Tokenizer

In [ ]:
# Save final model and tokenizer for deployment
final_model_dir = f"{MODEL_DIR}/legalbert_final_model"
!mkdir -p "{final_model_dir}"

# Save model
model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

# Save configuration
config_dict = {
    **CONFIG,
    'label_names': LABEL_NAMES,
    'class_weights': CLASS_WEIGHTS.tolist(),
    'best_val_f1': float(best_val_f1),
    'test_macro_f1': float(test_metrics['macro_f1']),
    'test_micro_f1': float(test_metrics['micro_f1']),
    'training_duration': str(training_duration)
}

with open(f"{final_model_dir}/config.json", 'w') as f:
    json.dump(config_dict, f, indent=2)

print("✓ Final model and tokenizer saved!")
print(f"  Location: {final_model_dir}")

## 15. Generate Final Report

In [ ]:
# Generate comprehensive final report
report = f"""
{'='*80}
CLAUSSIFIER - STAGE 1 (BERT RISK DETECTOR) TRAINING REPORT
{'='*80}

DATE: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
TRAINING DURATION: {training_duration}

{'='*80}
1. CONFIGURATION
{'='*80}
Model: {CONFIG['model_name']}
Number of labels: {CONFIG['num_labels']}
Max sequence length: {CONFIG['max_length']}
Batch size: {CONFIG['batch_size']}
Learning rate: {CONFIG['learning_rate']}
Number of epochs: {CONFIG['num_epochs']}
Device: {CONFIG['device']}

{'='*80}
2. DATASET
{'='*80}
Training examples: {len(train_dataset)}
Validation examples: {len(val_dataset)}
Test examples: {len(test_dataset)}
Total examples: {len(train_dataset) + len(val_dataset) + len(test_dataset)}

{'='*80}
3. TRAINING RESULTS
{'='*80}
Best Validation F1: {best_val_f1:.4f}
Final Train Loss: {history['train_loss'][-1]:.4f}
Final Val Loss: {history['val_loss'][-1]:.4f}

{'='*80}
4. TEST SET PERFORMANCE
{'='*80}
Test Loss: {test_metrics['loss']:.4f}

Macro Metrics:
  F1:        {test_metrics['macro_f1']:.4f}
  Precision: {test_metrics['macro_precision']:.4f}
  Recall:    {test_metrics['macro_recall']:.4f}

Micro Metrics:
  F1:        {test_metrics['micro_f1']:.4f}
  Precision: {test_metrics['micro_precision']:.4f}
  Recall:    {test_metrics['micro_recall']:.4f}

{'='*80}
5. PER-CLASS PERFORMANCE (TEST SET)
{'='*80}
"""

for i, label in enumerate(LABEL_NAMES):
    report += f"""
{i+1}. {label}
   F1:        {test_metrics['per_class_f1'][i]:.4f}
   Precision: {test_metrics['per_class_precision'][i]:.4f}
   Recall:    {test_metrics['per_class_recall'][i]:.4f}
"""

report += f"""
{'='*80}
6. SUCCESS CRITERIA
{'='*80}
Target Macro F1 ≥ 0.65: {'✓ PASS' if test_metrics['macro_f1'] >= 0.65 else '✗ FAIL'} ({test_metrics['macro_f1']:.4f})
Target Precision ≥ 0.70: {'✓ PASS' if test_metrics['macro_precision'] >= 0.70 else '✗ FAIL'} ({test_metrics['macro_precision']:.4f})
Target Recall ≥ 0.60: {'✓ PASS' if test_metrics['macro_recall'] >= 0.60 else '✗ FAIL'} ({test_metrics['macro_recall']:.4f})

{'='*80}
7. KEY INSIGHTS
{'='*80}
- Model successfully trained on {len(train_dataset)} examples
- Class-weighted loss function helped handle imbalance
- Best performance achieved at epoch {checkpoint['epoch'] if checkpoints else CONFIG['num_epochs']}
- Model ready for Stage 2 (Explainer) integration

{'='*80}
8. FILES GENERATED
{'='*80}
- Model checkpoint: {MODEL_DIR}/best_model_*.pt
- Final model: {final_model_dir}
- Training curves: {RESULTS_DIR}/training_curves.png
- Per-class performance: {RESULTS_DIR}/per_class_performance.csv
- This report: {RESULTS_DIR}/stage1_training_report.txt

{'='*80}
END OF REPORT
{'='*80}
"""

print(report)

# Save report
with open(f'{RESULTS_DIR}/stage1_training_report.txt', 'w') as f:
    f.write(report)

print("\n✓ Final report saved!")

**All files saved to Google Drive:**
- `/MyDrive/Claussifier/models/best_model_*.pt`
- `/MyDrive/Claussifier/models/legalbert_final_model/`
- `/MyDrive/Claussifier/results/training_curves.png`
- `/MyDrive/Claussifier/results/per_class_performance.csv`
- `/MyDrive/Claussifier/results/stage1_training_report.txt`